<a href="https://colab.research.google.com/github/zaky100/Zack-Air-Quality-Task/blob/main/Air_Quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Name: Zakaria Ahmed**

**Student Number: 20341433**

Course Code:

Professor:



# **Task 1: Data Selection & Handling**

**Loading Datasets**

These specific files belong to the PRSA (Beijing Multi-Site Air-Quality Data) dataset. Each file contains exactly four years of hourly readings (from March 1, 2013, to February 28, 2017) for a specific monitoring station in Beijing. They contain chemical pollution levels (like PM2.5, NO2) and weather data (like Temperature and Wind Speed).

In [1]:
import pandas as pd

# Load datasets
df_changping = pd.read_csv('PRSA_Data_Changping_20130301-20170228 (1).csv')
df_dingling = pd.read_csv('PRSA_Data_Dingling_20130301-20170228 (1).csv')
df_dongsi = pd.read_csv('PRSA_Data_Dongsi_20130301-20170228 (1).csv')
df_tiantan = pd.read_csv('PRSA_Data_Tiantan_20130301-20170228 (1).csv')

**Assign spatial categories**

This is a crucial data science technique. Because we are about to mix all four datasets together into one giant table, we need a way to remember which environment each row came from. If we didn't add these tags before mixing them, we wouldn't be able to easily compare the pollution of the city center versus the suburbs later on.

In [2]:
# Assign spatial categories
df_changping['category'] = 'Outer (Suburban)'
df_dingling['category'] = 'Outer (Suburban)'
df_dongsi['category'] = 'Inner (Urban)'
df_tiantan['category'] = 'Inner (Urban)'

**Merging & Processing**

**Stacks the Data**: It vertically merges the four separate station datasets into one continuous master table.

**Builds a Proper Timeline**: It takes the fragmented time data (year, month, day, hour) and fuses it into a single, standard datetime column—which is absolutely required for time-series forecasting—and deletes the messy leftovers.

**Organizes and Saves**: It reorders the columns so the most important identifiers (Time, Station, Category) are pushed to the very front for easy reading, and then saves this polished master dataset to your computer as a new CSV file ready for analysis.

In [3]:

# Merge dataframes
df = pd.concat([df_changping, df_dingling, df_dongsi, df_tiantan], ignore_index=True)

# Process datetime
df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
df.drop(columns=['No', 'year', 'month', 'day', 'hour'], inplace=True)

# Reorder columns
front_cols = ['datetime', 'station', 'category']
remaining_cols = [col for col in df.columns if col not in front_cols]
df = df[front_cols + remaining_cols]

# Save output
df.to_csv('Task1_Combined_Data.csv', index=False)

# **Task 2.1: Data Understanding**


Before running complex models, my first goal was to understand the dataset's shape and health. The statistical summary revealed that PM2.5 is highly right-skewed; its maximum recorded value is drastically higher than its 75th percentile, clearly indicating extreme pollution spikes. I also noted several missing values primarily concentrated in the pollutant columns, which is expected due to routine sensor maintenance.

In [4]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

print("--- 2.1 DATA UNDERSTANDING ---")

print(f"\n1. Dataset Shape:\nTotal Rows: {df.shape[0]}\nTotal Columns: {df.shape[1]}")

print("\n2. Column Descriptions & Data Types:")
df.info()

print("\n3. Missing Values Count (Pre-cleaning):")
missing_vals = df.isnull().sum()
print(missing_vals[missing_vals > 0])

print("\n4. Statistical Summary of Numerical Variables:")
display(df.describe().round(2))

--- 2.1 DATA UNDERSTANDING ---

1. Dataset Shape:
Total Rows: 140256
Total Columns: 15

2. Column Descriptions & Data Types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140256 entries, 0 to 140255
Data columns (total 15 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   datetime  140256 non-null  datetime64[ns]
 1   station   140256 non-null  object        
 2   category  140256 non-null  object        
 3   PM2.5     137276 non-null  float64       
 4   PM10      137868 non-null  float64       
 5   SO2       137117 non-null  float64       
 6   NO2       136010 non-null  float64       
 7   CO        132400 non-null  float64       
 8   O3        136931 non-null  float64       
 9   TEMP      140110 non-null  float64       
 10  PRES      140116 non-null  float64       
 11  DEWP      140110 non-null  float64       
 12  RAIN      140114 non-null  float64       
 13  wd        139820 non-null  object        
 14  WSPM   

,datetime,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM
count,140256,137276.00,137868.00,137117.00,136010.00,132400.00,136931.00,140110.00,140116.00,140110.00,140114.00,140142.00
mean,2015-03-01 11:29:59.999999744,76.37,98.78,14.91,44.66,1170.75,59.89,13.68,1010.15,1.98,0.06,1.86
min,2013-03-01 00:00:00,2.00,2.00,0.29,1.03,100.00,0.21,-16.80,982.40,-35.30,0.00,0.00
25%,2014-03-01 05:45:00,18.00,34.00,2.00,20.00,500.00,15.00,3.30,1001.60,-9.60,0.00,1.00
50%,2015-03-01 11:30:00,51.00,76.00,7.00,37.00,800.00,49.00,14.60,1009.80,2.40,0.00,1.50
75%,2016-02-29 17:15:00,107.00,136.00,18.00,63.00,1500.00,84.00,23.40,1018.50,14.60,0.00,2.30
max,2017-02-28 23:00:00,882.00,999.00,310.00,258.00,10000.00,1071.00,41.40,1042.00,28.80,52.10,10.50
std,NaN,78.68,88.63,20.22,32.34,1110.30,56.57,11.41,10.52,13.82,0.77,1.30


**Output Explanation 2.1:**

**Inference — Data Understanding & Diagnostics**


**1. Dataset Structure**

The combined dataset contains exactly 140,256 hourly observations across 15 columns. The data types are correctly formatted for machine learning: time is a proper datetime64 object, categories are text, and sensor readings are decimal numbers.

**2. Missing Values Analysis**

There is a clear difference in sensor reliability. Meteorological instruments (Temperature, Rain) are highly stable with almost zero missing data (less than 0.1%). Chemical pollutant sensors (PM2.5, CO) have up to 5.6% missing readings. This is a normal physical reality, as delicate chemical sensors require regular downtime for recalibration and maintenance.

**3. Statistical Insights**

The summary statistics reveal that PM2.5 levels are heavily right-skewed. While 75% of the readings fall below 107 µg/m³, the absolute maximum spikes to a hazardous 882 µg/m³. This massive gap confirms that the region does not suffer from consistently terrible air, but rather experiences sudden, extreme outlier smog events.



# **Task 2.2: Data Preprocessing**

To prepare the data, I had to handle the missing sensor values. Because this is time-series data, simply deleting rows would ruin the chronological order. Instead, I used linear interpolation to mathematically estimate missing values. I also engineered new features, extracting the "Month" and "Hour" for seasonal tracking, and creating an "AQI Level" column to translate raw PM2.5 numbers into readable health warnings.

**Handling Missing Values**

Because this is time-series data, dropping rows with missing values would break the chronological timeline, and filling them with an overall average would flatten out natural environmental spikes.

Instead, we use Linear Interpolation. This method mathematically estimates missing values by drawing a straight line between the last known reading and the next available reading. This safely fills the gaps while perfectly preserving the natural, continuous rise and fall of both weather and pollution trends over time. (Note: limit_direction='both' ensures gaps at the very extreme edges of the dataset are also filled).

In [5]:
print("--- 2.2 DATA PREPROCESSING ---")

# 1. Handling Missing Values via Interpolation
cols_to_interpolate = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
df[cols_to_interpolate] = df[cols_to_interpolate].interpolate(method='linear', limit_direction='both')

--- 2.2 DATA PREPROCESSING ---
